# HateXplain Formatting Notebook

This notebook formats **HateXplain** into the shared post-level schema used by downstream preprocessing and audit steps.

## What this notebook does
1. Loads HateXplain JSON from local disk.
2. Normalizes text and target fields.
3. Computes `binary_hate` using a positive-only majority rule from annotator labels.
4. Produces one standardized row per post.
5. Saves output to `outputs/preprocessing/01_hatexplain_standardized.tsv`.

## Output schema
- `post_id`
- `text`
- `raw_label`
- `binary_hate`
- `targets`
- `dataset`
- `text_dedup_key`

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional
import json
import re

import pandas as pd

# Resolve a reliable workspace root for both VS Code and CLI notebook runs.
WORKDIR = Path.cwd()
if not (WORKDIR / 'data').exists():
    WORKDIR = Path('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles')

@dataclass
class HatexplainConfig:
    # Input data location.
    hatexplain_path: Path = WORKDIR / 'data' / 'hatexplain.json'

    # Positive-only hate assignment: a post is 1 only when annotator hate share is strictly greater than this threshold.
    hatexplain_hate_majority_threshold: float = 0.5

    # Output location for standardized data.
    output_path: Path = WORKDIR / 'outputs' / 'preprocessing' / '01_hatexplain_standardized.tsv'

cfg = HatexplainConfig()
cfg.output_path.parent.mkdir(parents=True, exist_ok=True)
cfg

In [ ]:
def read_hatexplain_json(path: Path) -> pd.DataFrame:
    """Read HateXplain from either list-of-records or dict-of-records JSON formats."""
    if not path.exists():
        raise FileNotFoundError(f'Missing HateXplain file: {path}')

    with open(path, 'r', encoding='utf-8') as f:
        obj = json.load(f)

    if isinstance(obj, list):
        rows = obj
    elif isinstance(obj, dict):
        rows = []
        for key, value in obj.items():
            if isinstance(value, dict):
                row = value.copy()
                row.setdefault('post_id', key)
                rows.append(row)
    else:
        raise ValueError('Unsupported HateXplain JSON structure.')

    return pd.DataFrame(rows)


def normalize_text_for_dedup(text: str) -> str:
    """Create a lowercase, whitespace-normalized text key for dedup fallback."""
    if not isinstance(text, str):
        return ''
    return re.sub(r'\s+', ' ', text.strip().lower())


def normalize_target_token(token: str) -> Optional[str]:
    """Normalize one target token to a stable lowercase form."""
    if token is None:
        return None
    value = str(token).strip().lower().replace('_', ' ')
    if not value or value in {'none', 'null', 'nan'}:
        return None
    return value


def parse_targets_generic(value) -> List[str]:
    """Normalize mixed target payloads into a clean list[str]."""
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        candidates = list(value)
    elif isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = json.loads(s.replace("'", '"'))
                candidates = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                candidates = [s]
        elif '|' in s:
            candidates = [x.strip() for x in s.split('|')]
        elif ',' in s:
            candidates = [x.strip() for x in s.split(',')]
        else:
            candidates = [s]
    else:
        try:
            if pd.isna(value):
                return []
        except Exception:
            pass
        candidates = [value]

    normalized = []
    for candidate in candidates:
        token = normalize_target_token(candidate)
        if token:
            normalized.append(token)
    return sorted(set(normalized))


def serialize_targets_for_tsv(value) -> str:
    """Serialize list-like targets as pipe-delimited strings for TSV output."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    if isinstance(value, list):
        return '|'.join(str(x) for x in value)
    return str(value)


def serialize_any_for_tsv(value) -> str:
    """Serialize nested values so raw labels remain inspectable in flat files."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    if isinstance(value, (list, dict)):
        return json.dumps(value, ensure_ascii=False)
    return str(value)

In [ ]:
def harmonize_hatexplain(df: pd.DataFrame, hate_threshold: float) -> pd.DataFrame:
    """Transform HateXplain rows into the shared harmonized schema."""
    out = pd.DataFrame()

    # Keep stable post IDs when present; otherwise generate deterministic fallback IDs from row index.
    if 'post_id' in df.columns:
        out['post_id'] = df['post_id'].astype(str)
    elif 'id' in df.columns:
        out['post_id'] = df['id'].astype(str)
    else:
        out['post_id'] = pd.Series(df.index).astype(str)

    # HateXplain text can arrive as a full text string or token list.
    if 'text' in df.columns:
        out['text'] = df['text'].astype(str)
    elif 'post_tokens' in df.columns:
        out['text'] = df['post_tokens'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))
    else:
        raise KeyError('HateXplain text not found. Expected text or post_tokens.')

    def ann_to_binary(annotators) -> Optional[int]:
        # Positive-only rule: only strong hate-majority posts are marked 1.
        if not isinstance(annotators, list):
            return None

        labels = []
        for annotator in annotators:
            if not isinstance(annotator, dict):
                continue
            label = annotator.get('label')
            if label is None or pd.isna(label):
                continue
            labels.append(str(label).strip().lower())

        if not labels:
            return None

        hate_votes = sum(label == 'hatespeech' for label in labels)
        hate_share = hate_votes / len(labels)
        return 1 if hate_share > hate_threshold else 0

    def ann_to_targets(annotators) -> List[str]:
        # Collect union of target mentions across all annotators for a post.
        if not isinstance(annotators, list):
            return []
        all_targets = []
        for annotator in annotators:
            if isinstance(annotator, dict):
                all_targets.extend(parse_targets_generic(annotator.get('target')))
        return sorted(set(all_targets))

    if 'annotators' in df.columns:
        out['raw_label'] = df['annotators']
        out['binary_hate'] = df['annotators'].apply(ann_to_binary).astype('Int64')
        out['targets'] = df['annotators'].apply(ann_to_targets)
    elif 'label' in df.columns:
        # Fallback branch for alternate HateXplain exports that flatten labels.
        out['raw_label'] = df['label']
        out['binary_hate'] = pd.Series([pd.NA] * len(df), dtype='Int64')
        out['targets'] = df['targets'].apply(parse_targets_generic) if 'targets' in df.columns else [[] for _ in range(len(df))]
    else:
        out['raw_label'] = None
        out['binary_hate'] = pd.Series([pd.NA] * len(df), dtype='Int64')
        out['targets'] = [[] for _ in range(len(df))]

    out['dataset'] = 'hatexplain'
    out['text_dedup_key'] = out['text'].apply(normalize_text_for_dedup)

    # Keep one row per post ID so this output is directly comparable with other standardized datasets.
    out = out.drop_duplicates(subset=['post_id'], keep='first').copy()
    return out


def dataframe_for_tsv(df: pd.DataFrame) -> pd.DataFrame:
    """Convert nested/list columns into stable string-safe TSV representations."""
    export_df = df.copy()
    export_df['targets'] = export_df['targets'].apply(serialize_targets_for_tsv)
    export_df['raw_label'] = export_df['raw_label'].apply(serialize_any_for_tsv)
    return export_df

In [ ]:
hx_raw = read_hatexplain_json(cfg.hatexplain_path)
hx_std = harmonize_hatexplain(hx_raw, hate_threshold=cfg.hatexplain_hate_majority_threshold)
hx_std_tsv = dataframe_for_tsv(hx_std)

hx_std_tsv.to_csv(cfg.output_path, sep='\t', index=False)

print('Saved HateXplain standardized output to:', cfg.output_path)
print('Rows written:', len(hx_std_tsv))
display(hx_std.head(3))

In [ ]:
# Compare target-label distributions side by side:
# 1) all rows
# 2) rows where binary_hate == 1
# Targets are pipe-delimited strings in `hx_std_tsv['targets']`.

import matplotlib.pyplot as plt
from IPython.display import HTML, display


def build_target_distribution(df: pd.DataFrame, targets_col: str = 'targets') -> pd.DataFrame:
    """Return count/percentage distribution from pipe-delimited target strings."""
    target_series = df[targets_col].fillna('').astype(str)

    flat_labels = []
    for row in target_series:
        if not row.strip():
            continue
        parts = [p.strip() for p in row.split('|') if p.strip()]
        flat_labels.extend(parts)

    if not flat_labels:
        return pd.DataFrame(columns=['target_label', 'count', 'percentage'])

    dist = pd.Series(flat_labels).value_counts().rename_axis('target_label').reset_index(name='count')
    dist['percentage'] = (dist['count'] / dist['count'].sum() * 100).round(2)
    return dist


# Ensure binary_hate is numeric and nullable-safe for filtering.
work_df = hx_std_tsv.copy()
work_df['binary_hate'] = pd.to_numeric(work_df['binary_hate'], errors='coerce').astype('Int64')

all_dist = build_target_distribution(work_df)
hate_only_df = work_df[work_df['binary_hate'] == 1].copy()
hate_dist = build_target_distribution(hate_only_df)

# Build side-by-side tables for direct comparison.
all_display = all_dist.copy()
all_display.columns = ['target_label', 'count_all', 'percentage_all']

hate_display = hate_dist.copy()
hate_display.columns = ['target_label', 'count_hate_only', 'percentage_hate_only']

comparison_table = (
    all_display
    .merge(hate_display, on='target_label', how='outer')
    .fillna(0)
)
comparison_table['count_all'] = comparison_table['count_all'].astype(int)
comparison_table['count_hate_only'] = comparison_table['count_hate_only'].astype(int)
comparison_table = comparison_table.sort_values(['count_all', 'count_hate_only'], ascending=False).reset_index(drop=True)

print('Side-by-side target label comparison table:')
display(comparison_table)

# Optional visual with two separate tables shown side by side.
all_html = all_dist.to_html(index=False)
hate_html = hate_dist.to_html(index=False)
display(HTML(
    '<div style="display:flex; gap:24px; align-items:flex-start;">'
    '<div style="flex:1;"><h4>All Rows</h4>' + all_html + '</div>'
    '<div style="flex:1;"><h4>binary_hate == 1</h4>' + hate_html + '</div>'
    '</div>'
))

# Side-by-side pie charts for quick composition comparison.
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

if not all_dist.empty:
    axes[0].pie(
        all_dist['count'],
        labels=all_dist['target_label'],
        autopct='%1.1f%%',
        startangle=140,
        pctdistance=0.8,
    )
    axes[0].set_title('All Rows: Target Label Distribution')
else:
    axes[0].text(0.5, 0.5, 'No target labels', ha='center', va='center')
    axes[0].set_title('All Rows: Target Label Distribution')

if not hate_dist.empty:
    axes[1].pie(
        hate_dist['count'],
        labels=hate_dist['target_label'],
        autopct='%1.1f%%',
        startangle=140,
        pctdistance=0.8,
    )
    axes[1].set_title('binary_hate == 1: Target Label Distribution')
else:
    axes[1].text(0.5, 0.5, 'No target labels', ha='center', va='center')
    axes[1].set_title('binary_hate == 1: Target Label Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Change plot: compare target-label share in the original dataset (all rows)
# against the filtered subset (rows where binary_hate == 1).
# Positive values mean a label is more prevalent after filtering.
# Negative values mean a label is less prevalent after filtering.

# Merge distributions so each label has both baseline and filtered counts.
shift_df = all_dist.merge(
    hate_dist,
    on='target_label',
    suffixes=('_all', '_filtered'),
    how='outer'
).fillna(0)

# Use safe denominators to avoid divide-by-zero when a side has no labels.
all_total = shift_df['count_all'].sum()
filtered_total = shift_df['count_filtered'].sum()

shift_df['pct_all'] = (shift_df['count_all'] / all_total * 100) if all_total else 0.0
shift_df['pct_filtered'] = (shift_df['count_filtered'] / filtered_total * 100) if filtered_total else 0.0

# Delta is measured in percentage points: filtered minus original baseline.
shift_df['delta_pct_points'] = shift_df['pct_filtered'] - shift_df['pct_all']
shift_df = shift_df.sort_values('delta_pct_points')

# Color code: blue = increased share after filtering, gray = decreased share.
bar_colors = ['#0f90f2' if x > 0 else '#3b4046' for x in shift_df['delta_pct_points']]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(shift_df['target_label'], shift_df['delta_pct_points'], color=bar_colors)

# Zero line is the baseline where filtered share equals original share.
ax.axvline(0, color='black', linewidth=1.0)
ax.bar_label(bars, fmt='%+.1f pp', padding=4)

# Clear axis labels and title for interpretation.
ax.set_xlabel('Change in Share (percentage points): filtered - original')
ax.set_ylabel('Target label')
ax.set_title('Target Label Shift After Filtering to binary_hate == 1', pad=22)

# Subtitle clarifies directionality and comparison groups.
ax.text(
    0.5,
    1.02,
    'Baseline = all rows in HateXplain standardized output | Filtered = rows with binary_hate == 1',
    transform=ax.transAxes,
    ha='center',
    va='bottom',
    fontsize=10,
    color='#444444'
)

plt.tight_layout()
plt.show()

# Display the numeric shift table used to generate the chart.
display(shift_df[['target_label', 'count_all', 'pct_all', 'count_filtered', 'pct_filtered', 'delta_pct_points']])